# 🧬 Remedia — Reseptör Odaklı İlaç Keşif Pipeline'ı (tek notebook, Colab)

Bu notebook, tüm keşif akışını **baştan sona Google Colab'da** çalıştırır:
hedef protein → bağlanma cebi → tohum moleküller → yeni molekül üretimi →
**GNINA (GPU) docking** → ADMET → sıralama → görsel sonuç. **Hiç dosya
taşımak, hiç git senkronizasyonu yapmak yok.**

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mehmetg06/Remedia/blob/main/notebooks/remedia_pipeline.ipynb)

### 📋 Nasıl kullanılır — 3 kural
1. Üstten **Runtime ▸ Change runtime type ▸ Hardware accelerator ▸ GPU (T4)** seç.
   **GNINA docking (Hücre 5) GPU olmadan çalışmaz — GPU ZORUNLUDUR.**
2. **Runtime ▸ Run all** (veya her hücrede `Shift+Enter`, yukarıdan aşağıya).
3. Her hücrenin çıktısında **`✅`** işaretini gör; bir sonraki hücre bir
   öncekinin çıktısını (değişkenlerini) kullanır, sırayı bozma.

> Her hücrenin üstünde ne yaptığını, ne kadar süreceğini ve devam etmeden önce
> neyi görmen gerektiğini yazan bir not var.

## 🔵 Hücre 1 — Kurulum
**Ne yapıyor:** GNINA (GPU'lu docking binary'si), fpocket (cep tespiti), RDKit,
meeko ve gerekli Python paketlerini kurar; `src/` modüllerini import edebilmek
için Remedia reposunu klonlayıp `src/`'yi `sys.path`'e ekler; GPU'yu kontrol eder.
**Ne kadar sürer:** ~2–4 dakika (GNINA binary'si büyüktür, bir kere iner).
**Devam etmeden önce:** En sonda `✅ Kurulum tamam` yazısını ve GPU satırında
`bulundu ✔` gör. `⚠️ GPU BULUNAMADI` görürsen menüden GPU seçip bu hücreyi
TEKRAR çalıştır.

In [ ]:
import os, sys, stat, subprocess, urllib.request, shutil, datetime
from pathlib import Path

# --- 1) Remedia reposunu klonla (src/ modüllerini import edebilmek için) ----
REPO_URL = "https://github.com/mehmetg06/Remedia.git"
REPO_DIR = Path("Remedia")
if not REPO_DIR.is_dir():
    subprocess.run(["git", "clone", "-q", REPO_URL], check=True)
    print("• Remedia reposu klonlandi (Remedia/)")
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    print("• Remedia reposu guncel")

SRC_DIR  = (REPO_DIR / "src").resolve()
DATA_DIR = (REPO_DIR / "data").resolve()
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
print(f"• src/ import yoluna eklendi: {SRC_DIR}")

# --- 2) Python paketleri ----------------------------------------------------
def pip_install(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=False)

try:
    import rdkit  # noqa: F401
    print("• RDKit zaten kurulu")
except ImportError:
    print("• RDKit kuruluyor..."); pip_install("rdkit")

for mod, pkg in [("meeko", "meeko"), ("requests", "requests"),
                 ("pandas", "pandas"), ("tqdm", "tqdm")]:
    try:
        __import__(mod)
    except ImportError:
        print(f"• {pkg} kuruluyor..."); pip_install(pkg)

# --- 3) fpocket (baglanma cebi tespiti) -------------------------------------
if shutil.which("fpocket") is None:
    print("• fpocket kuruluyor (apt)...")
    subprocess.run("apt-get -qq update >/dev/null 2>&1", shell=True)
    subprocess.run("apt-get -qq install -y fpocket >/dev/null 2>&1", shell=True)
print("• fpocket:", shutil.which("fpocket") or "BULUNAMADI (Hucre 2 geometrik merkeze duser)")

# --- 4) GNINA GPU binary'si -------------------------------------------------
GNINA_PATH = "/usr/local/bin/gnina"
GNINA_URLS = [
    "https://github.com/gnina/gnina/releases/download/v1.3/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.1/gnina",
    "https://github.com/gnina/gnina/releases/download/v1.0.3/gnina",
]
if os.path.exists(GNINA_PATH) and os.path.getsize(GNINA_PATH) > 1_000_000:
    print("• GNINA zaten indirilmis")
else:
    subprocess.run("apt-get -qq install -y libboost-all-dev >/dev/null 2>&1", shell=True)
    ok = False
    for url in GNINA_URLS:
        try:
            print(f"• GNINA indiriliyor: {url}")
            urllib.request.urlretrieve(url, GNINA_PATH)
            if os.path.getsize(GNINA_PATH) > 1_000_000:
                ok = True; break
        except Exception as e:
            print(f"  (bu surum alinamadi: {e})")
    if not ok:
        raise RuntimeError("GNINA binary'si indirilemedi — bu hucreyi tekrar calistir.")
    m = os.stat(GNINA_PATH).st_mode
    os.chmod(GNINA_PATH, m | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)

ver = subprocess.run([GNINA_PATH, "--version"], capture_output=True, text=True)
print("• GNINA:", ((ver.stdout or "") + (ver.stderr or "")).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "kuruldu")

# --- 5) GPU kontrolu --------------------------------------------------------
gpu = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
if gpu.returncode == 0 and "NVIDIA-SMI" in (gpu.stdout or ""):
    print("• GPU: bulundu ✔")
else:
    print("⚠️ GPU BULUNAMADI! Runtime > Change runtime type > GPU (T4) sec, "
          "sonra BU HUCREYI TEKRAR calistir. GNINA docking (Hucre 5) GPU ZORUNLU.")

# --- 6) Ortak yollar (tum hucreler bunlari kullanir) ------------------------
RUN_ID = datetime.datetime.now().strftime("run_%Y%m%d_%H%M%S")
RESULTS_DIR = Path("results") / RUN_ID
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"\n✅ Kurulum tamam. RUN_ID={RUN_ID}  ·  sonuc klasoru: {RESULTS_DIR}")

## 🔵 Hücre 2 — Hedef: yapı indir + bağlanma cebi bul
**Ne yapıyor:** `UNIPROT_ID` için AlphaFold yapısını **REST API'den** indirir
(`fetch_structure.py` mantığı — sabit URL değil), `fpocket` ile en yüksek
druggability'li bağlanma cebini bulup merkezini otomatik hesaplar.
**Ne kadar sürer:** ~30–60 saniye.
**Devam etmeden önce:** `✅ HEDEF HAZIR` satırında reseptör yolunu ve docking
kutusu merkezini (Å) gör. Hedefi değiştirmek istersen sadece `UNIPROT_ID`'yi düzelt.

In [ ]:
UNIPROT_ID = "P00918"          # ← Hedef (varsayilan: Karbonik Anhidraz II). Ornek: P30405
BOX_SIZE   = (20.0, 20.0, 20.0)  # docking kutusu boyutu (A)

import fetch_structure, pocket_detection

# 1) AlphaFold yapisini REST API'den indir (sabit URL DEGIL)
pdb_path = fetch_structure.fetch_alphafold(UNIPROT_ID)
RECEPTOR = str(pdb_path)
print(f"• Reseptor yapisi: {RECEPTOR}")

# 2) fpocket ile en yuksek druggability'li cebi bul, merkezini hesapla
CENTER = None
if shutil.which("fpocket"):
    try:
        pocket = pocket_detection.best_druggable_pocket(pdb_path)
        CENTER = tuple(round(c, 3) for c in pocket["center"])
        print(f"• En iyi cep: Pocket {pocket['pocket_number']}  "
              f"druggability={pocket['druggability']}  hacim={pocket['volume']} A^3")
    except Exception as e:
        print(f"⚠️ fpocket cep bulamadi ({e}) — geometrik merkeze dusuluyor.")

# 3) Fallback: fpocket yoksa/basarisizsa tum atomlarin agirlik merkezi
if CENTER is None:
    xs = ys = zs = 0.0; n = 0
    for line in Path(pdb_path).read_text().splitlines():
        if line.startswith(("ATOM", "HETATM")):
            xs += float(line[30:38]); ys += float(line[38:46]); zs += float(line[46:54]); n += 1
    CENTER = (round(xs / n, 3), round(ys / n, 3), round(zs / n, 3))
    print("• Geometrik merkez kullanildi (fpocket yok).")

print(f"\n✅ HEDEF HAZIR")
print(f"   reseptor = {RECEPTOR}")
print(f"   docking kutusu merkezi (A) = {CENTER}   ·   boyut = {BOX_SIZE}")

## 🔵 Hücre 3 — Tohum moleküller
**Ne yapıyor:** `known_ligands.py` ile ChEMBL/PubChem'den hedefe ait bilinen
inhibitörleri otomatik getirir. Bulamazsa (veya kendi setini denemek istersen)
`MANUAL_SEEDS` üçlü-tırnaklı metnindeki SMILES'leri kullanır — form widget'ı
YOK, çok satırlı girdi sorunu yaşamazsın.
**Ne kadar sürer:** ~5–15 saniye.
**Devam etmeden önce:** `✅ N geçerli tohum molekül hazır` satırını ve listelenen
tohumları gör.

In [ ]:
from known_ligands import fetch_known_ligands
from rdkit import Chem

# ChEMBL bos donerse ya da kendi tohumlarini denemek istersen buraya yaz.
# Her satir: SMILES [isim].  # ile baslayan satirlar yok sayilir.
MANUAL_SEEDS = """
NS(=O)(=O)c1ccccc1                benzenesulfonamide
CC(=O)Nc1nnc(s1)S(N)(=O)=O        acetazolamide
"""

def parse_seed_text(text):
    out = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("#"):
            continue
        out.append(line.split()[0])
    return out

# 1) ChEMBL/PubChem'den otomatik getir
ligands, msg = fetch_known_ligands(UNIPROT_ID, max_results=5)
print(msg)

if ligands:
    seeds = [l["smiles"] for l in ligands]
    for l in ligands:
        print(f"  • {l['name']}: {l['smiles']}  ({l.get('activity', '')})")
else:
    seeds = parse_seed_text(MANUAL_SEEDS)
    print("• Elle girilen tohumlar (MANUAL_SEEDS) kullaniliyor:")
    for s in seeds:
        print("  •", s)

# Guvenlik agi + gecerlilik suzgeci
if not seeds:
    seeds = parse_seed_text(MANUAL_SEEDS) or ["NS(=O)(=O)c1ccccc1"]
seeds = [s for s in seeds if Chem.MolFromSmiles(s) is not None]

print(f"\n✅ {len(seeds)} gecerli tohum molekul hazir.")

## 🔵 Hücre 4 — Molekül üret (füzyon motoru)
**Ne yapıyor:** `molecule_generator.py`'nin füzyon motorunu çalıştırır (geniş
keşif → ön eleme → genetik optimizasyon → rafinasyon). GA fitness'i olarak
docking yerine QED (ilaç-benzerlik) vekilini kullanır (`docking_opts=None`) —
**gerçek docking bir sonraki hücrede GNINA/GPU ile ayrı yapılır**, çünkü Vina
kurulu değildir. Havuzu `GENERATE_N`'e tamamlamak için keşif molekülleri eklenir.
**Ne kadar sürer:** ~30–90 saniye (CPU'da).
**Devam etmeden önce:** `✅ N molekül üretildi` satırını ve örnek molekülleri gör.
Bu moleküller (`molecules`) sonraki hücrelerde docklanır.

In [ ]:
from molecule_generator import (
    fusion_generation, random_mutation, brics_recombination, write_smi,
)

GENERATE_N = 20   # docking'e girecek hedef molekul sayisi

# Fuzyon motoru: docking_opts=None -> GA fitness'i QED vekili (Vina gerekmez).
final, mode = fusion_generation(seeds, docking_opts=None, log_fn=print)
generated = [smi for smi, _ in final]

# Fuzyon top-5 dondurur; havuzu kesif molekulleriyle GENERATE_N'e tamamla
if len(generated) < GENERATE_N:
    pool = set(generated)
    pool.update(random_mutation(seeds, n=GENERATE_N))
    pool.update(brics_recombination(seeds, n=GENERATE_N))
    generated = list(pool)[:GENERATE_N]

# (isim, SMILES) listesi — tum sonraki hucreler bunu kullanir
molecules = [(f"mol_{i:03d}", smi) for i, smi in enumerate(generated, 1)]

# Kayit icin .smi yaz (molecule_generator.write_smi formatinda)
smi_out = RESULTS_DIR / "generated.smi"
write_smi(generated, smi_out, prefix="mol")

print(f"\n✅ {len(molecules)} molekul uretildi  ->  {smi_out}")
for name, smi in molecules[:10]:
    print(f"  {name}: {smi}")

## 🔵 Hücre 5 — GNINA DOCKING (GPU)
**Ne yapıyor:** Üretilen her molekülü RDKit ile 3D'ye hazırlar (`ligand_prep.py`
mantığı), sonra **GNINA'yı GPU'da CNN rescoring** ile çalıştırıp bağlanma
enerjisini (affinity, kcal/mol) hesaplar. Çıktı: `ligand, affinity_kcal_mol`
sütunlu bir DataFrame (`docking_df`).
**Ne kadar sürer:** molekül başına ~15–60 sn (T4). İlerleme çubuğu kalan süreyi
gösterir.
**Devam etmeden önce:** Her molekül için `✅ [isim]: affinity=X kcal/mol` gör.
Bir molekül `❌` verirse sorun değil — boş skorla raporlanır, diğerleri devam eder.
Daha negatif affinity = daha güçlü bağlanma.

In [ ]:
import re, time, subprocess
import pandas as pd
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm.auto import tqdm

DOCK_DIR = RESULTS_DIR / "gnina_out"
DOCK_DIR.mkdir(parents=True, exist_ok=True)


def prepare_ligand_sdf(smiles, name):
    # ligand_prep.py ile AYNI mantik: RDKit 3D + MMFF optimize, SDF yaz.
    # GNINA SDF'i dogrudan okur (Vina'daki ayri PDBQT adimina gerek yok).
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None
    mol = Chem.AddHs(mol)
    if AllChem.EmbedMolecule(mol, randomSeed=42) != 0:
        return None
    try:
        AllChem.MMFFOptimizeMolecule(mol)
    except Exception:
        pass
    sdf = DOCK_DIR / f"{name}.sdf"
    w = Chem.SDWriter(str(sdf)); w.write(mol); w.close()
    return sdf


def parse_affinity(out_sdf, stdout):
    # En iyi (1. siradaki) pozun affinity'sini kcal/mol dondurur:
    # once cikti SDF property'lerine, olmazsa stdout tablosuna bakar.
    try:
        for m in Chem.SDMolSupplier(str(out_sdf), removeHs=False):
            if m is None:
                continue
            for key in ("minimizedAffinity", "CNNaffinity", "affinity"):
                if m.HasProp(key):
                    return float(m.GetProp(key))
            break
    except Exception:
        pass
    for line in stdout.splitlines():
        mt = re.match(r"\s*1\s+(-?\d+\.\d+)", line)
        if mt:
            return float(mt.group(1))
    return None


rows = []
n = len(molecules)
print(f"{n} molekul GNINA (GPU · CNN rescoring) ile docklanacak.\n")
start = time.time()
bar = tqdm(molecules, desc="GNINA", unit="mol")
for idx, (name, smiles) in enumerate(bar, 1):
    sdf = prepare_ligand_sdf(smiles, name)
    if sdf is None:
        print(f"❌ {name}: gecersiz SMILES / 3D uretilemedi — atlandi")
        rows.append({"ligand": name, "affinity_kcal_mol": None})
        continue

    out_sdf = DOCK_DIR / f"{name}_docked.sdf"
    cmd = [
        GNINA_PATH, "-r", RECEPTOR, "-l", str(sdf),
        "--center_x", str(CENTER[0]), "--center_y", str(CENTER[1]), "--center_z", str(CENTER[2]),
        "--size_x", str(BOX_SIZE[0]), "--size_y", str(BOX_SIZE[1]), "--size_z", str(BOX_SIZE[2]),
        "--cnn_scoring", "rescore", "--seed", "42", "-o", str(out_sdf),
    ]
    proc = subprocess.run(cmd, capture_output=True, text=True)
    aff = parse_affinity(out_sdf, proc.stdout)

    if aff is not None:
        print(f"✅ {name}: affinity={aff:.2f} kcal/mol")
    else:
        tail = (proc.stderr or proc.stdout or "").strip().splitlines()
        print(f"❌ {name}: docking basarisiz — {' | '.join(tail[-2:]) if tail else 'bilinmeyen hata'}")
    rows.append({"ligand": name, "affinity_kcal_mol": aff})

    kalan = (time.time() - start) / idx * (n - idx) / 60.0
    bar.set_postfix_str(f"{idx}/{n} · tahmini kalan: {kalan:.1f} dk")

docking_df = pd.DataFrame(rows, columns=["ligand", "affinity_kcal_mol"])
ok = int(docking_df["affinity_kcal_mol"].notna().sum())
print(f"\n✅ GNINA bitti: {ok}/{n} molekul skorlandi.")
docking_df

## 🔵 Hücre 6 — ADMET (Lipinski / Veber)
**Ne yapıyor:** `admet_filter.py`'nin `lipinski_veber_filter`'ı ile her molekülün
MW, LogP, HBD/HBA, dönebilir bağ ve TPSA değerlerini hesaplayıp ilaç-benzerlik
filtresini uygular. Çıktı: `admet_df` DataFrame'i (`pass` sütunu geçti/kaldı).
**Ne kadar sürer:** ~1–5 saniye (offline, RDKit).
**Devam etmeden önce:** `✅ ... filtresini geçti` özet satırını gör.

In [ ]:
import admet_filter
import pandas as pd

admet_rows = [admet_filter.lipinski_veber_filter(smi, name) for name, smi in molecules]
admet_df = pd.DataFrame(admet_rows)

passed = int(admet_df["pass"].sum())
print(f"✅ ADMET: {passed}/{len(admet_df)} molekul Lipinski/Veber filtresini gecti.")
admet_df

## 🔵 Hücre 7 — Sırala (rank_report.py ile birleşik sıralama)
**Ne yapıyor:** Docking skorlarını ve ADMET sonuçlarını `rank_report.py`'nin
beklediği CSV formatlarında yazar, sonra `src/rank_report.py`'yi doğrudan çağırıp
birleşik sıralamayı üretir (önce ADMET'i geçenler, sonra en negatif affinity).
Çıktı: `ranked_df` ve `results/<RUN_ID>/final_ranking.csv`.
**Ne kadar sürer:** ~2 saniye.
**Devam etmeden önce:** `✅ Birleşik sıralama hazır` satırını ve sıralı tabloyu gör.

In [ ]:
import subprocess, sys
import pandas as pd

docking_csv = RESULTS_DIR / "docking_scores.csv"
admet_csv   = RESULTS_DIR / "admet_results.csv"
ranked_csv  = RESULTS_DIR / "final_ranking.csv"

# rank_report.py'nin okuyacagi tam formatlarda yaz
docking_df.to_csv(docking_csv, index=False)   # ligand, affinity_kcal_mol
admet_df.to_csv(admet_csv, index=False)       # ligand, pass, MW, LogP, violations, ...

# src/rank_report.py'yi dogrudan cagir
subprocess.run([
    sys.executable, str(SRC_DIR / "rank_report.py"),
    "--docking", str(docking_csv),
    "--admet", str(admet_csv),
    "--output", str(ranked_csv),
], check=True)

ranked_df = pd.read_csv(ranked_csv)
print(f"\n✅ Birlesik siralama hazir  ->  {ranked_csv}")
ranked_df

## 🔵 Hücre 8 — Sonuç: tablo + 2D çizim + Google Drive'a kaydet
**Ne yapıyor:** En iyi adayları tablo ve **RDKit 2D çizimleriyle** gösterir; tüm
sonuç dosyalarını (üretilen SMILES, docking/ADMET/sıralama CSV'leri, çizim PNG'si)
mount edilmiş **Google Drive**'da tarihli bir klasöre kaydeder — oturum kapansa
bile sonuçlar durur.
**Ne kadar sürer:** ~10–20 saniye (Drive mount izni ister).
**Devam etmeden önce:** En iyi aday tablosunu ve molekül çizimlerini gör; `✅ Tüm
sonuçlar Google Drive'a kaydedildi` satırını gör. Drive'a izin vermezsen sonuçlar
yine oturum içindeki `results/<RUN_ID>/` klasöründedir.

In [ ]:
import shutil
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Draw

TOP_K = min(6, len(ranked_df))
top = ranked_df.head(TOP_K).copy()
print("🏆 EN IYI ADAYLAR:")
try:
    display(top)
except NameError:
    print(top.to_string(index=False))

# 2D cizimler — ligand ismini molecules uzerinden SMILES'e esle
smi_by_name = {name: smi for name, smi in molecules}
mols, legends = [], []
for _, r in top.iterrows():
    smi = smi_by_name.get(r["ligand"])
    m = Chem.MolFromSmiles(smi) if smi else None
    if m is None:
        continue
    aff = r["affinity_kcal_mol"]
    aff_s = f"{aff:.2f}" if pd.notna(aff) else "—"
    admet_ok = "✓" if bool(r.get("admet_pass", False)) else "✗"
    mols.append(m)
    legends.append(f"{r['ligand']}  {aff_s} kcal/mol  ADMET={admet_ok}")

grid_path = RESULTS_DIR / "top_candidates.png"
if mols:
    grid = Draw.MolsToGridImage(mols, legends=legends, molsPerRow=3, subImgSize=(300, 250))
    grid.save(str(grid_path))
    try:
        display(grid)
    except NameError:
        print(f"(cizim kaydedildi: {grid_path})")

# --- Google Drive'a kalici kaydet (oturum kapansa bile durur) ---------------
saved = [docking_csv, admet_csv, ranked_csv, RESULTS_DIR / "generated.smi"]
if grid_path.exists():
    saved.append(grid_path)
try:
    from google.colab import drive
    drive.mount("/content/drive")
    save_dir = Path("/content/drive/MyDrive/Remedia_results") / RUN_ID
    save_dir.mkdir(parents=True, exist_ok=True)
    for f in saved:
        if Path(f).exists():
            shutil.copy(f, save_dir / Path(f).name)
    print(f"\n✅ Tum sonuclar Google Drive'a kaydedildi: {save_dir}")
except Exception as e:
    print(f"\nℹ️ Google Drive'a kaydedilemedi ({e}).")
    print(f"   Sonuclar oturum icinde hazir: {RESULTS_DIR.resolve()}")

print("\n🎉 PIPELINE TAMAMLANDI.")